Copyright 2026 Tampere University<br>
This notebook and software was developed for a Tampere University course COMP.CS.320.<br>
This source code is licensed under the MIT license. See LICENSE in the exercise repository root directory.<br>
Author(s): Ville Heikkilä \([ville.heikkila@tuni.fi](mailto:ville.heikkila@tuni.fi))

# COMP.CS.320 Data-Intensive Programming, Exercise 2

This exercise contains basic tasks of data processing using Spark and DataFrames. The tasks 1-5 concern a weather dataset, tasks 6-7 data on Netflix titles, and the last task is a theory question related to lecture content.

This is the **Python** version, switch to the Scala version if you want to do the tasks in Scala.

Each task has its own cell(s) for the code. Add your solutions to the cells. You are free to add more cells if you feel it is necessary.<br>
There are cells with test code following most of the tasks that involve producing code.<br>
Example outputs for the tasks can be found in a separate notebook: [Exercise 2 example outputs](https://adb-735222077226806.6.azuredatabricks.net/editor/notebooks/4422310316839943?o=735222077226806)

At the end of the notebook, there is a question regarding the use of AI or other collaboration when working the tasks.<br>
Please remember to answer the AI question. And finally, don't forget to submit your solutions to Moodle: [Weekly Exercise #2](https://moodle.tuni.fi/mod/assign/view.php?id=4162925)

##### Some resources that can help with the tasks in this exercise:

- The [tutorial notebook](https://adb-735222077226806.6.azuredatabricks.net/editor/notebooks/3816292345602665) from our course
- Chapter 3 in [Learning Spark, 2nd Edition](https://learning.oreilly.com/library/view/learning-spark-2nd/9781492050032/)
    - There are additional code examples in the related [GitHub repository](https://github.com/databricks/LearningSparkV2).
    - The book related notebooks can be imported to Databricks by choosing `import` in your workspace and using the URL<br> `https://github.com/databricks/LearningSparkV2/blob/master/notebooks/LearningSparkv2.dbc`
- [Databricks tutorial](https://docs.databricks.com/en/getting-started/dataframes.html) of using Spark DataFrames
- [Apache Spark documentation](https://spark.apache.org/docs/4.0.4/sql-ref-functions.html) on all available functions that can be used on DataFrames.<br>
  The full [Spark Python functions API listing](https://spark.apache.org/docs/4.0.4/api/python/reference/pyspark.sql/functions.html) for the functions package might have some additional functions listed that have not been updated in the documentation.

In [0]:
# some imports that might be required in the tasks

# importing functions as F to not to override native Python functions
# sum() will be the native sum-function, F.sum() the function from Spark
from pyspark.sql import functions as F
from pyspark.sql import DataFrame
from pyspark.sql import Row

## Task 1 - Weather data and the first rows from DataFrame

In the [Shared container](https://portal.azure.com/#view/Microsoft_Azure_Storage/ContainerMenuBlade/~/overview/storageAccountId/%2Fsubscriptions%2Fe0c78478-e7f8-429c-a25f-015eae9f54bb%2FresourceGroups%2Ftuni-cs320-f2026-rg%2Fproviders%2FMicrosoft.Storage%2FstorageAccounts%2Ftunics320f2026gen2/path/shared/etag/%220x8DE33DC133306E1%22/defaultId//publicAccessVal/None) in the `exercises/ex2/weather` folder is file `nordics_weather.csv` that contains weather data from Finland, Sweden, and Norway in CSV format.

The data is based on a dataset from Kaggle: [Finland, Norway, and Sweden Weather Data 2015-2019](https://www.kaggle.com/datasets/adamwurdits/finland-norway-and-sweden-weather-data-20152019).
The Kaggle page has further descriptions on the data and the units used in the data.

Part 1:

- Read the data from the CSV file into DataFrame called `weatherDF`. Let Spark infer the schema for the data.<br>
  Note that the column separator in the CSV file is a semicolon (`;`) instead of the default comma.
- Print out the schema of the created DataFrame. Study the schema and compare it to the data in the CSV file. Do they match?

Part 2:

- Fetch the first six rows of the weather data frame and print their contents.

In [0]:
file = "wasbs://shared@tunics320f2026gen2.blob.core.windows.net/exercises/ex2/weather"
weatherDF: DataFrame = spark.read  \
  .option("header", "true") \
  .option("sep", ";") \
  .option("inferSchema", "true") \
  .csv(file)


In [0]:
# Code that prints out the schema for weatherDF
weatherDF.printSchema()

In [0]:
weatherSample: list[Row] = weatherDF.head(6)

In [0]:
print("The first six rows of the weather data:")
print(*[list(row.asDict().values()) for row in weatherSample], sep="\n")  # prints each Row to its own line

## Task 2 - Aggregated values

Find the lowest and highest temperatures from the weather data. \
Find also the average temperature over all measurements.

In [0]:
lowestTemp: float = weatherDF.select(F.min('temperature_min')).collect()[0][0]
highestTemp: float = weatherDF.select(F.max('temperature_max')).collect()[0][0]
averageTemp: float = weatherDF.select(F.avg('temperature_avg')).collect()[0][0]

In [0]:
print(f"The lowest temperature is {lowestTemp}")
print(f"The highest temperature is {highestTemp}")
print(f"The average temperature is {averageTemp}")

## Task 3 - Adding columns

Add new columns, `measurement_year` and `measurement_month`, to the weather DataFrame and print out the schema and the first three rows for the new DataFrame.

- The new `measurement_year` column should contain the year as an integer based on the value in column `date`.
- The new `measurement_month` column should contain the month as an integer based on the value in column `date`.

(These new columns can be used in tasks 4 and 5.)

Finally, write code that prints out the new schema and the values of the first three rows.

In [0]:
weatherDFWithNewColumns: DataFrame = weatherDF \
        .withColumn('measurement_year',
                    F.year('date')) \
        .withColumn('measurement_month',
                    F.month('date'))

In [0]:
# code that prints out the schema and the first three rows for weatherDFWithNewColumns
weatherDFWithNewColumns.printSchema()

print(*[list(row.asDict().values()) for row in weatherDFWithNewColumns.head(3)], sep="\n")

## Task 4 - Aggregated DataFrame

Find the lowest, the highest, and the average temperature for each year.

Sort the resulting DataFrame based on year so that the latest year is the first row in the DataFrame.

In [0]:
yearlyTemperatureDF: DataFrame = weatherDFWithNewColumns.groupBy('measurement_year')\
    .agg(F.max('temperature_max').alias('highest_temp'),F.min("temperature_min").alias('lowest_temp'),F.avg('temperature_avg').alias('avg_temp'))\
    .sort(F.col('measurement_year').desc())

In [0]:
yearlyTemperatureDF.show()

## Task 5 - Average measurements in Finland

Using the weather data, find out the average precipitation (in cm) and the average snow depth (in mm) for each month in Finland in year 2018.<br>
Round both of the averages to 2 decimals, and sort the resulting DataFrame such that January is in the first row.

In [0]:
df_2018 = weatherDFWithNewColumns.filter(F.col("measurement_year") ==2018).filter(F.col("country") == "Finland")
avgFinlandDF: DataFrame = df_2018.groupBy('measurement_month')\
    .agg(F.round(F.avg('precipitation'),2).alias('avg_precip_cm'),
         F.round(F.avg("snow_depth"),2).alias('avg_snow_depth_mm')) \
    .sort(F.col('measurement_month').asc())


In [0]:
avgFinlandDF.show()

## Task 6 - Netflix data

In the [Shared container](https://portal.azure.com/#view/Microsoft_Azure_Storage/ContainerMenuBlade/~/overview/storageAccountId/%2Fsubscriptions%2Fe0c78478-e7f8-429c-a25f-015eae9f54bb%2FresourceGroups%2Ftuni-cs320-f2026-rg%2Fproviders%2FMicrosoft.Storage%2FstorageAccounts%2Ftunics320f2026gen2/path/shared/etag/%220x8DE33DC133306E1%22/defaultId//publicAccessVal/None) in the `exercises/ex2/netflix` folder is file `netflix_titles_us_2022.csv` that contains information about the movie and TV shows that were available in Netflix streaming in United States at July 2022.

The data is from a dataset from Kaggle: [Netflix TV Shows and Movies](https://www.kaggle.com/datasets/victorsoeiro/netflix-tv-shows-and-movies).
The Kaggle page has the descriptions for the data.

**Part 1:**

- Read the data from the CSV file into data frame `titlesDF`.
- Only include columns about title name, type, description, release year, and runtime in the data frame.
- Note that in the CSV, the movie or show descriptions
    - can contain multiple lines
    - and double quotes (`"`) in the descriptions are marked with two double quote characters
        - for example, `bounty hunters (also referred to as ""Cowboys"")` \
          should be interpreted as `bounty hunters (also referred to as "Cowboys")`
- With the default settings, Spark CSV reader does not give the correct results. The available options related to reading in CSV files as DataFrames can be found at: [CSV file source options](https://spark.apache.org/docs/4.0.4/sql-data-sources-csv.html#data-source-option)

**Part 2:**

- Extract the description of the movie `Inception` to variable `inceptionDescription`.


In [0]:
file = "wasbs://shared@tunics320f2026gen2.blob.core.windows.net/exercises/ex2/netflix"
titlesDF: DataFrame = spark.read  \
  .option("header", "true") \
  .option("sep", ",") \
  .option("inferSchema", "true") \
    .option("multiLine", "true") \
  .option("escape", '"') \
  .csv(file)

titlesDF = titlesDF.select('title','type','description','release_year','runtime')

inceptionDescription: str = titlesDF.filter(F.col("title") == 'Inception').select("description").collect()[0][0]

In [0]:
titlesDF.printSchema()
titlesDF.show(2, truncate=True)
print(f"\nInception: {inceptionDescription}")

## Task 7 - Counting movies and shows

**Part 1:**

- Find out how many movies and how many TV shows are listed in the dataset.
- Assign the results to variables `movieCount` and `showCount`.

**Part 2:**

- Consider only the movies and shows that were released before 2010.
- Create a data frame, `titleCountDF`, that contains the movie and show count for the five years which have the most released titles in total.

In [0]:
# code for finding out how many movies (movieCount) and TV shows (showCount) are included in the dataset
movieCount = titlesDF.filter(F.col("type") == 'MOVIE').count()
showCount = titlesDF.filter(F.col("type") == 'SHOW').count()

# the movie and show counts for releases before 2010
titleCountDF: DataFrame = titlesDF.filter(F.col('release_year')<2010)\
    .groupBy("release_year")\
    .agg(F.count(F.when(F.col("type") == "MOVIE", 1)).alias("movie_count"),
         F.count(F.when(F.col("type") == "SHOW", 1)).alias("movie_count"))\
    .sort(F.count(F.col("title")), ascending=False).limit(5)

In [0]:
print(f"Movies in the dataset:   {movieCount}")
print(f"TV shows in the dataset: {showCount}")

In [0]:
titleCountDF.show()

## Task 8 - Theory questions

Using your own words, answer the following questions:

- Scaling:
    1. What is meant by "scaling" in the context of data engineering?
    2. What is the difference between horizontal and vertical scaling?
- In this exercise, you have been using Spark to do the exercise tasks.
    3. What is Spark, and why is it used in the course?
    4. In general, when can using Spark be beneficial?<br>
        (compared to, for example, handling the data processing with native programming language tools, or other libraries like Pandas)

Extensive answers are not required here.<br>
If your answers do not fit into one screen, you have likely written more than what was expected.

1. Scaling means increasing the processing power of your infrastructure to accommodate for increasingly large data
2. Horizontal scaling involves including more individual processing nodes (e.g. computers) into your infrastructure, vertical scaling means improving the computing power of a single node
3. Spark is a unified computing engine, allowing for easy combinations of different APIs and efficient data processing. The course deals with distributed processing of data practices and thinking, and Spark is perfect for it.
4. Spark has faster in-memory computing compared to MapReduce, use cases for it can range from a single laptop to clusters of thousands servers, it is versatile for both streaming and batch processing by handling streaming in micro-batches.

## Use of AI and collaboration

Using AI and collaborating with other students is allowed when doing the weekly exercises.<br>
However, the AI use and collaboration should be documented.

- Did you use AI tools while doing this exercise?
  - Did they help? And how did they help?
- Did you work with other students to complete the tasks?
  - Only extensive collaboration is expected to be reported. If you only got help for a couple of the tasks, you don't need to report it here.

I used AI to check my queries and give hints for functions because there are so many.